In [1]:
from pathlib import Path
import numpy as np 
from PIL import Image
import matplotlib.pyplot as plt 

DATA_DIR = Path(r"C:\data-mars\msl\ncam")
IMG_DIR = DATA_DIR / "images" / "edr"
ROVER_DIR = DATA_DIR / "images" / "mxy"
RANGE_DIR = DATA_DIR / "images" / "rng-30m"
TRAIN_LBL_DIR = DATA_DIR / "labels" / "train"
TEST_LBL_DIR = DATA_DIR / "labels" / "test" / "masked-gold-min3-100agree"

for d in [IMG_DIR, ROVER_DIR, RANGE_DIR, TRAIN_LBL_DIR, TEST_LBL_DIR]:
    print(d.name, d.exists())

edr True
mxy True
rng-30m True
train True
masked-gold-min3-100agree True


### 1. **Folder structure** 


In [ ]:
# exploring the names 

# searches the edr folder for every file ending in .JPG
# produces them one by one as a generator, and sorted(...) collects them into a sorted list of Path objects
all_imgs = sorted(IMG_DIR.glob("*.JPG"))  # gives a list of all image file paths 
#using "*.png" for labels and masks 
print(len(all_imgs))

# tuples: a folder, and the pattern to search it with
folders = [
    (IMG_DIR, "*.JPG"),
    (ROVER_DIR, "*.png"),
    (RANGE_DIR, "*.png"),
    (TRAIN_LBL_DIR, "*.png"),
    (TEST_LBL_DIR, "*.png"),
]

for d, pattern in folders:
    files = sorted(d.glob(pattern))
    print(d.name, len(files))
    for p in files[:3]: # p is one Path at a time, and p.name is its filename
        print("   ", p.name)

18127
edr 18127
    NLA_397586928EDR_F0010008AUT_04096M1.JPG
    NLA_397586934EDR_F0010008AUT_04096M1.JPG
    NLA_397681339EDR_F0020000AUT_04096M1.JPG
mxy 18097
    NLA_397586928MXY_F0010008AUT_04096M1.png
    NLA_397586934MXY_F0010008AUT_04096M1.png
    NLA_397681339MXY_F0020000AUT_04096M1.png
rng-30m 18096
    NLA_397586928RNG_F0010008AUT_04096M1.png
    NLA_397586934RNG_F0010008AUT_04096M1.png
    NLA_397681339RNG_F0020000AUT_04096M1.png
train 16064
    NLA_397586934EDR_F0010008AUT_04096M1.png
    NLA_397681339EDR_F0020000AUT_04096M1.png
    NLA_397681372EDR_F0020000AUT_04096M1.png
masked-gold-min3-100agree 322
    NLA_409036068EDR_F0051606NCAM00348M1_merged.png
    NLA_409036116EDR_F0051606NCAM00348M1_merged.png
    NLA_409036860EDR_F0051662NCAM00346M1_merged.png


The naming pattern, the key to matching:

Folder	Example name
image (edr)	NLA_397586934EDR_F0010008AUT_04096M1.JPG
train label	NLA_397586934EDR_F0010008AUT_04096M1.png
test label	NLA_409036068EDR_F0051606NCAM00348M1_merged.png
rover mask	NLA_397586934MXY_F0010008AUT_04096M1.png
range mask	NLA_397586934RNG_F0010008AUT_04096M1.png

So:
- Train label ↔ image: identical stem, only the extension differs.
- Test label ↔ image: same, but remove _merged first.
- Masks ↔ image: replace MXY or RNG with EDR.

(NLA = Navcam left camera A, and the long number is the spacecraft clock time when the photo was taken.)

In [4]:
img_stems = set()
for img in all_imgs:
    img_stems.add(img.stem)

train_stems = set()
for lbl in sorted(TRAIN_LBL_DIR.glob("*.png")):
    train_stems.add(lbl.stem)

test_stems = set() 
for lbl in sorted(TEST_LBL_DIR.glob("*.png")):
    test_stems.add(lbl.stem.replace("_merged", ""))

train_lbl_with_img = train_stems & img_stems
test_lbl_with_img = test_stems & img_stems
train_test_overlap = train_stems & test_stems #must be empty!
img_with_no_lbl = img_stems - train_stems - test_stems

print(len(train_lbl_with_img), "train labels with an image")
print(len(test_lbl_with_img), "test labels with an image")
print(len(train_test_overlap), "train_test overlap")
print(len(img_with_no_lbl), "images iwth no label")


16064 train labels with an image
322 test labels with an image
0 train_test overlap
1741 images iwth no label


### 2. **Sizes** 

### 3. **Label values** 
Label values (Navcam uses the “NAV” key):

- 0 soil, 1 bedrock, 2 sand, 3 big rock
- 255 = no label. This is what you’ll later pass to the loss as ignore_index=255, so unlabelled pixels don’t count.

Train vs test labels are different quality:

- Train: crowdsourced. A pixel keeps its label only if at least 3 people labelled it and 2/3 of them agreed; otherwise it’s set to 255.
- Test: labelled by experts with 100% agreement, in three versions (min1, min2, min3) depending on how many experts had to label each pixel. min3 is the sparsest but most reliable.
→ Decision suggestion: report results mainly on min3, the cleanest version, and optionally on the others too.

The rover and far-away areas are already masked in the labels: anything beyond 30 m and the rover itself are set to “no label”. The separate mask files are:

- MXY: rover mask (1 = rover)
- RNG-30M: range mask (1 = further than 30 m)

### 4. **Visualise** 

### 5. **Class balance** 

### 6. **Camera types** 
- ncam (Navcam): greyscale stereo images that the rover uses to drive.
- mcam (Mastcam): colour images for science and for human operators, and usually without range data (the distance information behind the “>30 m away” mask).

#### Decisions
1. Train on ncam (driving camera, has range masks, greyscale).
2. Later experiment: transfer to mcam.
